<a href="https://colab.research.google.com/github/33MarGomez/Interactive-Tutorials/blob/main/EPR_notes3_teacups_and_the_Liouvillian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Notes in EPR: teacups and the Liouville Space

(Part 3 of 3 in the series, *notes in EPR* )

by: Marco Gomez, initial release version: June 01, 2026

##Scope and Introduction

The installation lines are only as complicated as running a cell with the gitclone, then right-clicking copy-path in the files tab on the left and pasting in what you get.

teacups is written in a class-based programming format. The coverage is going to be fundamental to understanding how the program is written because learning physics this way is like adding hashtags when a definition is met. It's a new, very modern way to approach the material so these lessons will take full advantage of that without teaching introductory Python. The beginning sections are focused on the organization and calculations of teacups to aid development or examine the complexity of the calculation. Discussion on Liouville space is deferred to the end section.

You can use the Colab side-bar to jump between headings and the Github symbols interface to jump between functions for easier reading.

In [ ]:
!pip install -q condacolab
import condacolab
!git clone 'https://github.com/TheresiaQuintes/teacups.git'
import sys
sys.path.append("/content/teacups/src")

##Getting Started: The Workflow

Read the teacups paper [here](https://pubs.acs.org/doi/10.1021/acs.jpca.5c01512). The paper's supplementary information demonstrates exemplary inputs with little code, here is a quick reference of all the parameters you can give teacups as a pamphlet so distributed in its documentation [here](https://theresiaquintes.github.io/teacups/_downloads/7d69dfccbf540b0e705373a625109db3/quickreference.pdf), this is a link to the pdf that will not automatically download. As can be seen, the only module imports needed in your experiment are teacups.classes and teacups.simulations.

In [ ]:
import teacups.simulations as sim
import teacups.classes as cl

teacups.classes imports the three classes that collect system, experimental, and options for your simulations. teacups.simulations contents are one big function that makes the experiment go, the best starting point. All of teacups is written with strong function descriptions so you can quickly read through what something does at a surface level. Read it like a book- the front cover gives you what the simulation takes in and returns. As a class-based program, you might be surprised by returns: none with several attributes listed. When you declare in the function variable_name.some_class, that variable is assigned something within that class and then that gets called an attribute in the function description. So it tagged something when you see that, and you might need to refer to where that information came from, each time you're curious about an object's form. So Simulations is an important branching point for this kind of programming analysis. simulations starts with several module functions from teacups.input_handlers.

*   **input_object_handler**: The first line called. Its whole purpose is to create the cal "dump" where all completed values are placed, but beware this includes intermediate steps.
*   **scale_inputs**: The documentation asks for MHz, mT but must uses seconds-based experiments. All inputs converted to SI units; Hz and T.
*   **predefinitions**: this is where cal.spec_sim is created, containing the final spectrum. It also creates cal.t, counting every time step. Those lines are shown below, spec_sim is seen to be- upper dimension, *(number of time progressions)*, of lower dimension, *(divisions of the field axis)*. teacups will carry forward these upper two dimensions nearly all the time to structure the calculation. The immediate lower dimension, added by Multioperator or Multimatrix classes, is the grid points not done here.
```
cal.t = np.linspace(exp.t_scale[0], exp.t_scale[1], exp.t_points,
                        dtype=FLOAT_TYPE)
cal.spec_sim = np.zeros((exp.t_points, len(exp.B_z)), dtype=COMPLEX_TYPE)
```
*   **initialize_spin_system**: only assigns the $s=\sum_{n} m_s$ total unpaired spin that will be used for $S = 2s+1$ dimension matrices later. It does this as a python list in sys.s corresponding to the spins of systems. It takes the radical-pair 'rp' string you provided as the subject of study and gives it sys.s = [1/2,1/2]. If you designated 'trip', sys.s = [1]. 'tdp' triplet:doublet product pair systems have list sys.s = [1/2, 1] as their subject.
*   **create_grid**: checks the string for the grid you want and passes it on to relevant modules teacups.grid and teacups.epr_grid. These are cited from and code attributed to other authors, requiring minimal review. cal.theta and cal.phi of the rotation grid are assigned here as returns of the grid generating functions. *key*: the last line opt.grid_points = len(cal.phi) rounds the number of grid points to whatever the function returned, so you may need to check your reported sampling.
*   **split_grid**: uses the size of the density matrix at the end to divide angle lists into sub-lists to feed different computer cores to run the calculation faster.

Scroll to the bottom to check the function description's outputs. About halfway through sims.teacups(), checks if the user is using teacups to diagonalize a matrix. "if opt.eigval_mode is True" returns the list of population values as a 1-D numpy array at every field, at every field orientation. The reason why there is a later function, signals_and_processing.powder_average(), is because every hamiltonian runs through the Multioperator class that makes it a grid point and that function adds up the weakness of the field inductions from unaligned samples into the anisotropic details of the signal. In all other cases you run simulations.teacups() with returns corresponding to intensity and (optionally) time evolved populations, this return must be indexed with three dimensions, the bottom one is the list. It also ends the calculation without continuing to the other two returns, which is why we check returns.

There is one return condition, SimOpt.pop_evolution = True, because it designates evolution of the eigenfunctions which only occurs in Liouville space. Therefore, SimOpt.space == 'liouville', this second condition checks a bad user input. Across both spaces, a propagator is properly used and $\rho(t)$ is provided in cal.spec_sim, the pop_evolution condition is strictly change of basis in the upper space.

The entire middle of the function lays out the groundwork for how we read through teacups() so let's organize the rest of the lesson. Based on the split_grid of inputs_handler, we can work out the quantum mechanics happens in the for-loop. The next few lines confirm our suspicions, convolution.voigt_convolution() broadens the signal by FWHH and is the only part that calls error variables. As implied, these are standard deviations in seconds and mT by which the signal occurs in time (Gaussian) and is field extended in the Lorentzian. These use the scipy.ndimage.gaussian_filter() function, which is noted to apply the standard deviation along each axis individually [1] [2]. Note that in teacups documentation, this is listed as being in pixels, but this seems to be a deferral to the way it works by placing the signal wrong along the timeline to smoothly represent CW scanning, which would make the weighing in divisions of seconds in that axis. This is particularly founded in SimOpt.extend_t extending the experiment timeline backwards to accomodate the extra width in smoothing the function. Note that teacups has the means to represent relaxation in the Liouville; this is purely to better match spectrometer data and make the curve less noisy. In Hilbert space, the next line calculates a non-physical signal decay so it is not even a phenomenological lattice time, which is specificed by the user in sys.decay.
```
cal.spec_sim = co.voigt_convolution(sys.sigma_time, sys.width_gauss,
                                        cal.spec_sim, opt.extend_t)

if SimOpt.space == 'hilbert':
        sap.signal_hilbert_decay(sys, cal)
```
The rest of this lesson deals with the lines in the middle, which specify the Hamiltonian and render the signal, and are a roadmap for following the calculation.
*   **creators.set_up_spinoperator** Sets up the vector of matrices $[S_x,S_y,S_z]$ based on 2S + 1.
*   **creators.set_up_observable** If in Hilbert space, it grabs the $S_y$ from the recently created spin operator. This is the observation direction, and will be multiplied with $\rho(t)$ in the trace for populations.
*   **creators.set_up_tensors** Sets up all the hamiltonian operators that make up the hamiltonian, such as calculating the g-factor anisotropy matrix.
*   **hamiltonians.set_up_{system}_hamiltonian** Sets cal.ham_sys, the appropriate spin number hamiltonian after reading sys.s in an if statement.
*   cal.ham = cal.ham_sys + ham.set_up_mw_hamiltonian, adds the above system hamiltonian to the irradiation contribution to $B$.
*   **density_matrices.set_up_density_matrix** Calls on user parameters around initial population specification to construct the outer product.
*   **hamiltonians.set_up_commutator_superoperator** This constructs the superoperator prior to its exponentiation
*   **signals_and_processing.propagation** Builds the $e^{\pm Ht}$ that modifies populations in Hilbert space or exponentiates the superoperator built in the previous line.
*   **signals_and_processing.make_signal** Applies the observable and propagator
*   **signals_and_processing.powder_average** Adds up all the gridpoints given to the intensity.

As the final map towards where things point, here are the imports at the top of each module, to refer to what code you may need to learn, and to build a high-level picture of the physical concepts that are organized in each and how deep any code could go.

*   **simulations** signals_and_processing, teacups.input_handler, hyperfine, convolution, creators, hamiltonians, density_matrices
*   **input_handler** grid (-> epr_grid)
*   **hamiltonians** multioperator_tools, matrix_tools (-> orientation_dependent_ham), relaxation
*   **density_matrices** multioperator_tools, creators, hamiltonians
*   **creators** matrix_tools (-> orientation_dependent_ham)

teacups.memory in density_matrices allocates cores in parallel computing because it is the identified bottleneck by input_handlers.split_grid(), whose module also imports it. All the rest of the modules rely on the code detailed in their body wholly.

[1]. The Scipy Community. *gaussian_filter - SciPy 1.17.0* https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html

[2]. Wikimedia. *Weierstrass transform* https://en.wikipedia.org/wiki/Weierstrass_transform


##The Multioperators and Matrix classes

These contain various manipulations that size matrix. The goal is to demonstrate that every concept from the last two lessons is as it looks, with high quality simulations possible by 4 by 4 and 16 by 16 matrices translated to anisotropic 3 by 3 operators. All Hilbert space matrices are in the triplet x,y,z basis set like those of the real solutions to the p-orbitals. This section covers the classes that construct quantum mechanical observables to aid in development, and to advance the idea that only a few geometry-focused lines of Python can build book-quality code.

When creators.set_up_spin_operator encounters sys.s, which is a list of $m_s$ values, it breaks it up and assigns it the matrix\_tools.SpinOperator class, whose \_\_init\_\_ argument takes the first spin and one other spin optionally as a secondary coupling. SpinOperator(Operator) means it is an object of the Operator class and Operator(Matrix) means an Operator is an object of the Matrix class. So all operators are also matrices and descriptions of spin are operators. Matrix, the broadest class, only has an \_\_init\_\_ argument for its dimension and immediately writes a square matrix of zeroes (dimension, dimension) as self.matrix. SpinOperator is a special case of it, so does not provide its dimensionality to formulate its matrix until it figures out its dimensions. Matrices start this way in teacups, as zeros allocation in memory before being changed by quantum mechanical properties. In the case of SpinOperator, after building out its Pauli matrices, the last thing it does is assign self.matrix and self.dimension. The reason is that the inherited method (the functions written to the broadest class) include .scalar(), .product(), .basis_transformation() and they do not return anything while changing self.matrix data. For example, the contents of .scalar(self,multipliers) read self.matrix \*\= multipliers. As a child class, it's able to take creator's interpretation of sys.s, calculate $S = 2s+1$, apply ladder operators $S_{+}$ and $S_{-}$ recursively to render the Pauli matrices and the final result is written to S.matrix in module creators, and the length of the matrix to S.dimension. As an Operator object, in addition to everything that matrix can do, it also gains .build_vector() and .build_superoperator(). After assigning the matrix element, .SpinOperator runs self.build_vector() which gives it an attribute self.vector which is every row of the vector in a one dimension contiguous list. It then .build_superoperator() in the next line, regardless of 'liouville' setting, which makes $\mathbb{I}$ sized to self.dimension, and takes the left or right kroenecker to it. So when something is an Operator in teacups, it is also a matrix repeated on the diagonal as many times as its dimension squared. All operators are matrices and a spin operator is an operator.

The creators.S variable is set the result of matrix_tools.SpinOperator(Operator) is set to cal.s, that is, all the results of the SpinOperator class are eventually stored in cal.s. Variable S is named this way after the first line in SpinOperator \_\_init\_\_, but has one additional matrix associated with it, S.matrix_coupling_spins.
```
dim_spin = int(2*spin+1)
```
spin is the first entry of the list we fed it, if a second entry exists in rp or tdp, it is coupling_spins. To determine how big the final matrix is going to be, it sizes up the number of couplings and gives them $S = 2s + 1$
```
dim_couplings = 2*coupling_spins+1
dim_couplings_total = int(np.prod(dim_couplings))
dim_total = int(dim_spin*dim_couplings_total)
```
No matter what, it builds a standard $[S_x,S_y,S_z]$ set called Pauli_uncoupled. Its dimensions may not be what we expect when secondary couplings are introduced, but its elements are recognizable. Our first physical insight is that even in the case of interacting terms of the hamiltonian, zeeman splitting is treated as uncorrelated with some lower dimension, $\mathbb{I}$. This is explicitly mentioned in the paper as resulting from the secular approximation, $D_{zz} = ?$, $D_{...} = 0$.
```
pauli_spin = np.zeros((3, dim_total, dim_total),
                                  dtype=COMPLEX_TYPE)
pauli_uncoupled = self.pauli_matrices(spin)
dimension = int(2*spin+1)                                  ##this is dim_spin but in .pauli_matrices() body
for i in range(0, dimension-1):
  off_diagonal_elements[i] = np.sqrt((i+1)*(dimension-1-i))
for i in range(0, dimension):
  sigma_z[i, i] = dimension/2.0-i-0.5
  if i+1 <= dimension-1:
    sigma_x[i, i+1] = 0.5*off_diagonal_elements[i]
    sigma_y[i, i+1] = 0.5*1j*off_diagonal_elements[i]
sigma_x = sigma_x+np.transpose(np.conjugate(sigma_x))
sigma_y = sigma_y+np.transpose(np.conjugate(sigma_y))
sigma_y = np.conjugate(sigma_y)
pauli_spin = np.kron(pauli_uncoupled, np.eye(dim_couplings_total,
                                                         dtype=FLOAT_TYPE))
self.matrix = pauli_spin
```
In the final creators.S, "S.matrix = S.matrix + S.matrix_coupling_spins[0]" .This is similar to Hamiltonian descriptions such that $S_1^TS_2$ is added to terms containing $S_1$ and $S_2$ in, respectively, coupling or dipolar exchange versus zeeman interaction. This is only functional with one other coupling - when the paper was published, it only did 'tdp' and 'rp' as sys.spin_system options - which is why it only adds the first element of the coupling system. There is a for-loop sizing up the dimension running down the list but what is included in the following code-block illustrates that it makes the spin at that level and it is then at least smaller than dim_spin. This is the same as having n_coup = 1 in the code since n_coup always captures the first coupling. Some first approximation is possible this way, and the n_coup list organizes the coupling_spins dimension to always be lower the higher the spin's index.
```
number_of_couplings = len(coupling_spins)
pauli_coupling_spins = np.zeros((number_of_couplings,
                                             3, dim_total, dim_total),
                                            dtype=COMPLEX_TYPE)
for n in range(0, number_of_couplings):
  pauli_uncoupled = self.pauli_matrices(coupling_spins[n])
  pauli_tmp = pauli_uncoupled
  pauli_coupling_spins[n] = np.kron(
                        np.eye(dim_spin, dtype=FLOAT_TYPE), pauli_tmp)
```
So wherever things are fetched for the hamiltonian's eigenfunctions, will be from x,y,z as is detailed in the paper and transformed to the high-field triplet/singlet eigenfunction set from the non-eigenfunction (zero-field) retrievals. It is also possible to see that the previous lesson's operators $\hat{S}_1^T\hat{S}_2$ created an eigenvalue times a matrix, instead of a single number as was detailed. To clarify this detail, a basis transformation that will appear often is in the 'rp' where cal.s is turned into $|\alpha \alpha \rangle$, $\frac{1}{\sqrt{2}}(|\alpha\beta \rangle + |\beta\alpha \rangle)$,  $\frac{1}{\sqrt{2}}(|\alpha\beta \rangle - |\beta\alpha \rangle)$, and $|\beta \beta \rangle$
```
#/... of creators module
def set_up_observable(sys: object, opt: object, cal: object):
  obs = mt.Operator(cal.s.dimension)
  obs.matrix = cal.s.get('y')
  if sys.spin_system == 'rp':
    st_transformation = np.array([[1, 0, 0, 0],
                                [0, np.sqrt(1/2), np.sqrt(1/2), 0],
                                [0, -np.sqrt(1/2), np.sqrt(1/2), 0],
                                [0, 0, 0, 1]], dtype=FLOAT_TYPE)
    obs.matrix = st_transformation.T@obs.matrix@st_transformation
```

An equally important class is Multioperator, which encompasses all Hamiltonians and density matrices constructed by teacups. It takes Operator classes, and places them in dimensions according to B_angle_shape = (b_points, grid_points, self.dimension, self.dimension), and the syntax for Hamiltonian = mut.Multioperator(cal.s, opt.grid_points, exp.B_z*MU_B), tells of the lowest dimensions on paper being $\sigma_{x,y,z}$ which are $2s+1$ by $2s+1$ square matrices for five dimensions total. This is a data point per field strength, per crystal orientation, per quantum mechanical degree of freedom x,y,z, per S by S Pauli matrix. However, Multioperator.dimension is itself only the S value, it is the overlying object that represents any dimension as a bin, and uses SpinOperator.get() for individual pauli matrices in practice. There is no additional dimension to arrays, and operations like np.linalg.eigh, which calculates eigen properties, iterates over these higher dimensions automatically. This is the main importance placed on it, that it collects angles and fields for simulations.teacups, by doing it for hamiltonian matrices. The cal.s syntax passes through Multioperator(Multioperator\_) which inherits Multioperator\_(Multimatrix). Multioperator has useful functions outside of working with cal.s objects, because it contains several physical calculations, and Multioperator_ can be fed (dimension, grid_points, B_z) to build vectors and superoperators for any property. The class multioperator_tools.Multimatrix resembles matrix_tools.Matrix except for the additional dimensions, its purpose is that actions like scalar multiplication or basis transformation are applied universally by field strength and powder average.

```
def exchange_coupling(self, J_ex: float, spinop2: object) -> None:
  self.matrix = J_ex * (self.spinop.get('x')@spinop2.get('x')
                                + self.spinop.get('y')@spinop2.get('y')
                                + self.spinop.get('z')@spinop2.get('z'))
```
Despite its name, matrix_tools.Tensor(Matrix) class is only responsible for taking user inputs around the laboratory frame to simulate observation directions. All quantum mechanical tensor quantities mentioned in lesson 1 are Operators(Matrix) items and, more accurately, use Spin_Operator in some way.

##Implementation

This section collects key lines across teacups to show how quantities are calculated. Throughout this section, lines allocating memory will be included because they also allocate dimensionality. Its inclusion is useful for picking up a calculation after a bug, or for custom adoption. The observation follows the typical density formalism to observables as trace along the product of observation matrix and $\rho$.

```
#in creators
def set_up_observable(sys: object, opt: object, cal: object) -> None:
  obs = mt.Operator(cal.s.dimension)
  obs.matrix = cal.s.get('y')

#in signals_and_processing
def make_signal(exp: object, opt: object, cal: object) -> None:
  cal.signal = np.zeros((len(cal.t), len(exp.B_z), opt.grid_points),
                          dtype=COMPLEX_TYPE)

#in signals_and_processing
def time_evolution_hilbert(cal: object) -> "np.ndarray":
  propagation_invers = np.linalg.inv(cal.propagation)
  for i in tqdm(range(len(cal.t))):
      cal.signal[i] = np.trace(
                    (rho_prop @ cal.observable), axis1=-1, axis2=-2)
      rho_prop = propagation_invers @ rho_prop @ cal.propagation
```
The density matrix is developed as an outer product, though numpy makes it difficult to discern; np.linalg.eigh has its bottom dimension corresponding to individual associated eigenvalues, and next upper the vector itself. By switching these dimensions, the product is still rows of columns, but the elements have moved around to represent different components, so that such a mix will no longer generate an eigenvalue relationship with the hamiltonian. The final element exploits elementwise multiplication by python using '*' instead of '@'.
```
if sys.precursor == "zf":
        if sys.spin_system == "trip":
          rho = mut.Multioperator(cal.s, opt.grid_points, exp.B_z)
          ham_tri_hf = ham.set_up_triplet_high_field_xyz_hamiltonian(
                exp, opt, cal)
          eig_hf, vec_hf = np.linalg.eigh(ham_tri_hf)
          rho.B_angle_matrix = (
                np.conj(np.transpose((vec_hf), (0, 1, 3, 2)))
                @ np.diag(np.array(sys.population, dtype=FLOAT_TYPE))
                @ vec_hf
                )
          rho.B_angle_matrix *= np.eye(3, dtype=FLOAT_TYPE)
```
A useful case study might be how to approach differently sized systems that demonstrate cross-coupling, such as 'tdp'. Dimensionally, the paper describes that this will be the respective dimensions in a kronecker product. This is correct, the cancellation for the diagonal is sized '*=np.eye(6)'. As the paper describes, the +1, 0, -1 eigenbasis parameters are transformed into the x,y,z basis as expected, by creating a transformation matrix from the eigenvectors. Please note that the code comments are taken directly from the module - this part is well described.
```
#from hamiltonians
def set_up_tdp_hamiltonian(sys: object, exp: object, opt: object, cal: object) -> 'np.ndarray':
    s = cal.s

    cal.s = mt.Spinoperator(1)
    ham_tri = set_up_triplet_hamiltonian(exp, opt, cal)

    cal.s = mt.Spinoperator(1/2)
    ham_doub = set_up_doublet_hamiltonian(exp, opt, cal)

    cal.s = s
    ham_tdp = np.zeros((len(exp.B_z), opt.grid_points, 6, 6),
                       dtype=COMPLEX_TYPE)
    ham_tdp = np.kron(ham_doub, np.eye(3, dtype=FLOAT_TYPE)) \
        + np.kron(np.eye(2, dtype=FLOAT_TYPE), ham_tri)

#from density_matrices
def set_up_density_matrix(sys: object, exp: object, opt: object, cal: object):
  elif sys.precursor == "triplet-pnm":
    elif sys.spin_system == "tdp":
            # set up the full high field hamiltonian of the coupled system
            # in the pnm-Basis(+1, 0, -1): ppnnmm
            ham_pnm = ham.set_up_tdp_high_field_pnm_hamiltonian(
                sys, exp, opt, cal)

            # define rho in pnm-basis using populations for triplet-zf levels
            rho_trip = np.diag(np.array(sys.population[2:], dtype=FLOAT_TYPE))
            rho_trip = np.kron(rho_trip, np.eye(2, dtype=FLOAT_TYPE))

            rho = mut.Multioperator(cal.s, opt.grid_points, exp.B_z)
            rho.matrix = rho_trip

            # diagonalise high field hamiltonian to get the eigenvectors for
            # transformation pnm-basis <-> TDP-eigenbasis
            eig_hf, vec_hf = np.linalg.eigh(ham_pnm)

            # diagonalise the spin system hamiltonian to get the eigenvectors
            # for the transformation product basis <-> TDP-eigenbasis
            eig_sys, vec_sys = np.linalg.eigh(cal.ham_sys)

            # basistransformation rho: pnm-basis -> TDP-eigenbasis
            rho.B_angle_matrix = (
                np.conj(np.transpose((vec_hf), (0, 1, 3, 2)))
                @ rho.matrix
                @ vec_hf
            )
            rho.B_angle_matrix *= np.eye(6, dtype=FLOAT_TYPE)

            # basistransformation rho: TDP-eigenbasis -> product basis
            rho.B_angle_matrix = (
                vec_sys
                @ rho.B_angle_matrix
                @ np.conj(np.transpose((vec_sys), (0, 1, 3, 2)))
            )

            rho.B_angle_matrix *= np.eye(6, dtype=FLOAT_TYPE)

            # add density matrix for the doublet in product basis
            rho_doub = np.diag(np.array(sys.population[:2], dtype=FLOAT_TYPE))
            rho_doub = np.kron(rho_doub, np.eye(3, dtype=FLOAT_TYPE))
            rho.B_angle_matrix += rho_doub
```
The propagator is built normally, using the first time step as $\Delta t$ and exponentiating the eigenvalues to an argument.
```
#from signals_and_processing
def propagation(sys: object, opt: object, cal: object) -> None:
  step = cal.t[1]-cal.t[0]
  propagation = np.zeros(cal.ham.shape, dtype=COMPLEX_TYPE)

  eigval, vec = np.linalg.eigh(cal.ham)
  exp_arg = 1j*eigval

  n = propagation.shape[-1]
  propagation[:, :, range(n), range(n)] = np.exp(exp_arg*step)
  print('exponential ready...')

  propagation = vec @ propagation @ np.conj(
                  np.transpose(vec, (0, 1, 3, 2)))
```
To explore the Liouville space, begin with the shape of the density matrix. It is flattened with a new row starting each column - note that construction in ascending order of eigenvectors in numpy means that the vector is incrementing eigenvalue positivity.
```
#reminder from __init__ of Multimatrix
self.B_angle_shape = (b_points, grid_points,
                              self.dimension, self.dimension)

#from multioperator_tools
def matrix_changed(self) -> None:
self.B_angle_matrix = np.broadcast_to(
            self.matrix, self.B_angle_shape).copy()

#from multioperator_tools
def build_vector(self) -> None:
  self.B_angle_vector = self.B_angle_matrix.reshape(
            (self.B_angle_matrix.shape[0], self.B_angle_matrix.shape[1],
             self.dimension**2))

#from density_matrices
def set_up_density_matrix(sys, exp, opt, cal) -> None:
  if opt.space == "liouville":
        rho.build_vector()
        cal.rho = rho.B_angle_vector
```
Its propagation can be a little confusing without first going back for the dimensionality of the superoperator hamiltonian, which is a propagator. It is as it is in literature, finding the superoperator commutation relationship. It does not mix field or grid points, but it does square the length of individual dimensions without adding any new ones. The cal.eigvec is a usage of the return tuple for the eigenvectors of the Hilbert space hamiltonian, so this object is two dimensional. The propagation line creates a new axis of length 1 that when used with * two dimensional axis, creates a diagonal square matrix of eigenvalues. The reason this is done instead of using diag is memory efficiency. Therefore, the propagation transforms the eigenvectors of the superoperator by the exponentiation of the eigenvalue, this is a typical procedure requiring follow up on the one line of the shortcut. It then then takes the exponential matrix of the propagator and takes a product with the large cal.rho vector with an equally sized cal.observable vector.
```
#from creators
def set_up_observable(sys: object, opt: object, cal: object) -> None
  elif opt.space == 'liouville':
        obs.build_vector()
        cal.observable = obs.vector

#from hamiltonians
def set_up_commutator_superoperator(sys, opt, cal):
        ham_adj = np.transpose(np.conjugate(cal.ham), [0, 1, 3, 2])
        ham_superop = np.kron(np.eye(
            cal.ham.shape[-1], dtype=FLOAT_TYPE), cal.ham[:, :] ) - \
            np.kron(ham_adj[:, :], np.eye(cal.ham.shape[-1], dtype=FLOAT_TYPE))
        cal.ham_superop = ham_superop

#from signals_and_propagators
def propagation(sys: object, opt: object, cal: object) -> None:
  elif opt.space == 'liouville':
        step = cal.t[1]-cal.t[0]

        cal.ham_superop *= -1j*step
        print('superoperator ready...')

        eigval, vec = np.linalg.eig(cal.ham_superop)
        print('eigenvalues ready...')

        propagation = vec @ (np.exp(eigval)[:, :, np.newaxis] *
                             np.eye(eigval.shape[-1], dtype=COMPLEX_TYPE))\
            @ np.linalg.inv(vec)
        print('propagator ready...')
  cal.propagation = np.array(propagation)

#from signals_and_propagation
def time_evolution_liouville(cal: object) -> "np.ndarray":
    rho_prop = cal.rho[:, :, :, np.newaxis]
    for i in tqdm(range(0, len(cal.t))):
        cal.signal[i] = np.dot(rho_prop[:, :, :, 0], cal.observable)
        rho_prop = cal.propagation@rho_prop

```
For population evolutions, the dimension are turned into a square matrix again and become a one-dimensional list at the end. That is added to a running list and the matrix is propagated by the superoperator commutator.
```
#from simulations
def teacups(Sys, Exp, SimOpt) -> 'np.ndarray':
  if opt.space == 'liouville':
            cal.eigvec = np.linalg.eigh(cal.ham_sys)[1]

if opt.pop_evolution is True and opt.space == 'liouville':
        print("starting population evolution...")
        rho_prop = cal.rho[0, 0, :, np.newaxis]
        cal.pop_evolution = []
        for i in range(0, len(cal.t)):
            pop_matrix = np.conj(np.transpose(cal.eigvec[0, 0])) @\
                rho_prop.reshape((cal.s.dimension, cal.s.dimension)) @\
                cal.eigvec[0, 0]
            pops = np.diag(pop_matrix)
            cal.pop_evolution.append(pops)
            rho_prop = cal.propagation[0, 0]@rho_prop
        cal.pop_evolution = np.array(cal.pop_evolution)
```
Any tensor of interest can be examined for its structure in the module creators. The module multioperator_tools holds all the multioperator contributions of the hamiltonian including the exchange, microwave and zeeman couplings. Note that the cosines vector of field is being called an operator in such a case, the archecture of that section focuses mainly on calculating the quantity by field and grid point.

##Testing Environment

The imports I needed to bug test, or check individual lines were
```
import teacups.creators as cr
import teacups.input_handler as inputs
import teacups.classes as cl

sys, exp, opt, cal = inputs.input_object_handler(Sys, Exp, SimOpt)
inputs.initialize_spin_system(sys)
inputs.create_grid(opt, cal)
cr.set_up_spinoperator(sys, cal)
cr.set_up_tensors(sys, cal)
```
And everything has worked, importing individual modules or copy and pasting lines of code. The documentation's dev section is thankfully thorough about where objects are created, running the function that creates it will advance the code to the next object error if necessary (thankfully!)

##The Liouvillian and Liouville Space
Concise tutorials are provided by doctors Muhammed Sabieh Anwar and Jerryman A. Gyamfi, of which the latter so thorough that the peer review version is not necessary to verify claims [3][4][5]. Their generosity is thanked, and what is detailed here is a yet more concise summary, meeting the level of the teacups simulation.

teacups uses the Liouvillian for describing the evolution of populations in a time dependent capacity. The reason is that Liouville space is the only way to describe the change of Hilbert-space component vectors of the hamiltonian, which means it is also the only way to describe changes in its eigenvalues. A concise expression to this end is as follows. The double bracket indicates a vector in Liouville space, where the outer products $|a\rangle\langle b|$ making up the hamiltonian are placed into one long vector, $\sigma(t)$ whose base case $\sigma(0)$ is the $t=0$ density matrix.

$$|\sigma(t)\rangle\rangle = \widehat{\hat{U}}|\sigma(0)\rangle\rangle = exp(-i(\hat{H} \otimes\mathbb{I}-\mathbb{I}\otimes\hat{H})t/\hbar)|\sigma(0)\rangle\rangle$$
$$=exp(-i\hat{H}t/\hbar)\sigma(0)exp(-i\hat{H}t/\hbar)$$

This equation concludes the coverage of populations and energy in EPR, it was placed ahead of definitions of Liouville space as the master equation that motivates coverage, since it concisely summarizes motion in the space. We work backwards from this point, first showing that multiplication at a kronecker product level represents exponentiation as follows. Using dimension d of the Hilbert space via the completeness definition, and the kroenecker mixed product rule, the superoperator vector is the rows of a quantum mechanical matrix operator placed in sequence as a $1 \times d^2$ vector. This mapping is the Liouville space. The same product is represented two different ways, either as a deterministic matrix of coefficients made from individual $\langle \phi_n|\psi(t)\rangle$ inner products, or the equivalent column-row relationship $\langle \phi_n|\psi(t)\rangle\langle\psi(t)|\phi_{n'}\rangle$. The latter form might be recognized as what was done in lesson 1 around spin-orbit coupling, using the Hilbert space change-of-basis with $M_s$, with this step hidden. In fact, we may now see that any density matrix can be formulated as a product of two kronecker deltas making up coefficients, even if not diagonal [3].

$$|a\rangle\langle b| \mapsto |a\rangle \otimes | b\rangle^{*} \equiv |ab\rangle\rangle$$

$$\rho(t)=\mathbb{I_d}|\psi(t)\rangle\langle\psi(t)|\mathbb{I_d}=\sum_{n=1}^{d}\sum_{n'=1}^{d}|\phi_n\rangle\langle\phi_n|\psi(t)\rangle\langle\psi(t)|\phi_{n'}\rangle\langle\phi_{n'}| = \sum_{n=1}^{d}\sum_{n'=1}^{d} \rho_{n,n'}(t)|\phi_n\rangle \langle \phi_{n'}|$$

$$\equiv \sum_{n=1}^{d}\sum_{n'=1}^{d}c_{n,n'}(t)|\phi_n\rangle\langle\phi_{n'}|$$

$$ ( \langle \psi_{a'}(t)| \otimes \langle \psi_{b'}(t)|^{*}) ( |\psi_a(t)\rangle \otimes |\psi_b(t)\rangle^{*}) = \delta_{a,a'}\delta_{b,b'}$$

$$\mathbb{I_d}=\sum_{n=1}^{d} |\phi_n\rangle\langle \phi_n |$$

This is already the change of basis according to the spectral decomposition theorem that would imply the inner product between equivalent Hilbert spaces is equal to a change in eigenvalues.

$$\hat{A}=\sum^{n=1}_{d}\lambda_{n}|a_n\rangle\langle a_n|$$
$$\langle a_n|a_{n'} \rangle = \delta_{n,n'}$$

The relationship to the eigenfunctions is put simply another way, using the supercommutator and the mixed product kronecker product rule and the typical eigenvalue/eigenvector relationship. There is an assumed step where $|a\rangle \otimes |b\rangle^{*}$ is the representation of the lower product in space and the value created by the hermitian hamiltonian is real.

$$\widehat{\hat{U}} |ab \rangle \rangle = | (\hat{H}\otimes \mathbb{I})|a\rangle\langle b| - (\mathbb{I}\otimes\hat{H})|a\rangle\langle b| \rangle\rangle$$

$$=|\hbar\omega_{a}|a\rangle\langle b | - \hbar{\omega_b}|a\rangle\langle b| \rangle \rangle$$

$$=\hbar(\omega_a-\omega_b)|ab\rangle\rangle$$

teacups reshapes the vectors back into a square after undergoing evolution, then takes the dot product with the base-case hamiltonian, before applying another evolution of the supercommutator. This is only to help numpy with the finding of the eigenvalue in the appropriate basis.

The Liouville space is a description of the following problem. This product always exists because of the earlier $\rho(t)$ formulation for a scalar resulting from change of basis in Hilbert space. The intention is to construct operators from $|\nu' \rangle\langle \nu'|$ if they originally came from $|\nu \rangle\langle \nu|$. Matrices are recognized as operators in quantum mechanical space, so this represents changes to the eigenfunctions of operators themselves, changing their structure as time goes on. In teacups, not covered in the lesson but covered well in the paper, this revolves around population changes leading to a decay of transverse and logitudinal transfer mechanisms through natural energy level depletion. This is the basis for NMR experiment NOE and EPR equivalent ENDOR; such capability is powerful in representing zero-quantum transfer.

$$\hat{A}=\sum_{\nu=1}^{d}\sum_{\nu'=1}^{d}A_{\nu\nu'}|\nu\rangle\langle \nu'|$$

$$A_{\nu,\nu'} = \langle \nu |\hat{A}|\nu \rangle$$

Of the measurement vectors which must fulfill the following criterion that leads to a trace in the above expression of expectation values. Note this is the probability of finding the state m in a classical-scale sampling.

$$p(m)=\langle \psi|M_{m}^{\dagger}M_{m}|\psi\rangle$$

$$\langle \psi |\psi\rangle = 1 \Rightarrow \sum_m M_{m}^{\dagger}M_m = \mathbb{I_d}$$

And of course, in the following linear problem, the solution can be represented as a supervector as the left side of the sum rolls over in X-column multiplier sequence every row of A, and each row of X has the same repeated sequence of B-multipliers in the other part of the sum. So C is uniquely identified. $\tilde{D}$ is the special matrix that represents this- teacups has the ability to construct an equivalent operation called create_bilinear_operator in class Multioperator. How exciting!

$$AX + XB = C$$

$$\tilde{D} (\mathbb{vec}[X]) = \mathbb{vec}[C]$$

$$(\mathbb{vec}[X])=\tilde{D}^{-1}\mathbb{vec}[C]$$

$$\mathbb{vec}[|a\rangle\langle b|] = \mathbb{vec}
\begin{bmatrix}  a_{11}b_{11} & a_{11}b_{12} & a_{11}b_{13} & {...} \\
a_{21}b_{11} & a_{21}b_{12} & a_{31}b_{13} & ... \\ \vdots \end{bmatrix} = \begin{bmatrix} a_{11}b_{11} \\ a_{11}b_{12} \\ \vdots \\ a_{21}b_{11} \\ \vdots \\ a_{d1}b_{1d} \end{bmatrix} $$

[3]. Gyamfi, J. M. Fundamentals of Quantum Mechanics in Liouville Space, Ver. 3. *Eur. J. Phys* June, 2020. DOI: http://arxiv.org/abs/2003.11472v3

[4]. Anwar, M. S. *SUPEROPERATORS IN NMR: A SUMMARY.* https://physlab.lums.edu.pk/images/4/46/Superop.pdf

[5]. Jerryman A. Gyamfi 2020 Eur. J. Phys. 41 063002. DOI:
https://doi.org/10.1088/1361-6404/ab9fdd]